In [1]:
# =============================================================================
# CS-781 Phase 7 -- Prior Case Retrieval (PCR) with Arch #1 (decoder-only, Phase-1 model)
#
# Task: for every query case, rank the candidate cases of ITS OWN pool; submit exactly 7 unique ids (Micro-F1@7).
# Data: train 827 queries / 4,320 candidates, dev 118 / 1,023, test 237 / 1,727.
# Design
#   * Bi-encoder: a case -> up to N chunks of 1023 tokens (+<s>) -> backbone -> mean of token states per chunk ->
#     mean over chunks -> L2-normalised vector. Similarity = cosine.  (No pretrained model other than YOUR Phase-1 model.)
#   * Training: contrastive (InfoNCE, multi-positive) per query: positives = its relevant candidates (<= N_POS),
#     negatives = BM25-mined HARD negatives (N_HARD) + random negatives (N_RAND). BM25 is classical lexical matching on the
#     competition data only (no pretrained model) and is used (a) to mine hard negatives and (b) as an optional fusion channel.
#   * KD: the student (9 of 12 layers) matches the teacher's query->document similarity distribution (+ the contrastive loss).
#   * Evaluation: Micro-F1@7 exactly as the official scorer; the dev score picks the best epoch. Epoch 0 = the un-tuned
#     Phase-1 model (how well do Phase-1 representations transfer?).
# Techniques (TECHNIQUES): full | lorma | lora | adapter | partial | normtune | kd
# Kaggle: paste each "# %%" block as its own cell, in order. FIRST RUN: TECHNIQUES=["lorma"]; read the printed data statistics.
# NOT yet run on Kaggle -- BM25, metric and parsing logic were tested on fake data; the torch code is untested.
# =============================================================================


# %% [1] Imports & HF login
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, pickle, gc, traceback, functools, itertools
from collections import Counter

import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import CountVectorizer

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda


In [2]:
# %% [2] Config
TECHNIQUES = ["lorma"]        # e.g. ["lorma"], ["adapter"], ["full", "kd"]   (KD needs its teacher's pcr_best.pt: put 'full' before 'kd')
SEED = 42
SKIP_DONE = True
TEACHER_TECH = "full"
TEACHER_CKPT = ""             # KD only: path to the teacher's pcr_best.pt ("" = working dir, else it retrains the teacher)

ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/arch1final"                       # tokenizer (legal_bpe.json)
MODEL_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-model1/model_best.pt"       # Arch #1 Phase-1 checkpoint

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P7 = "Phase_7_PCR"
FILES = {"train": f"{P7}/Train/train.json", "train_c": f"{P7}/Train/train_candidates.json",
         "dev": f"{P7}/Dev/dev.json",       "dev_c": f"{P7}/Dev/dev_candidates.json",
         "test": f"{P7}/Test/test.json",    "test_c": f"{P7}/Test/test_candidates.json"}

K = 7                         # the scorer looks at the top 7 only
CHUNK_TOK = 1023              # tokens per chunk (+<s> = 1024 = the Phase-1 context)
TRAIN_CHUNKS = 2              # chunks per case while training (memory)
INFER_CHUNKS = 4              # chunks per case at evaluation / test time
EMB_MODE = "spread"           # which chunks of a long case: "spread" (evenly over the case) | "head" (first ones)
MAX_DOC_WORDS = 12000         # very long cases: first HEAD_WORDS words + the tail
HEAD_WORDS = 6000

N_POS, N_HARD, N_RAND = 3, 4, 1   # per training query: positives / BM25 hard negatives / random negatives
HARD_POOL = 40                    # hard negatives are sampled from the top-40 BM25 non-relevant candidates
TAU = 0.05                        # contrastive temperature (cosine / TAU)
ACCUM = 4                         # queries per optimiser step
EVAL_CHUNK_BATCH = 16             # chunks per forward pass at evaluation
WEIGHT_DECAY, WARMUP_FRAC, DROPOUT_FT = 0.01, 0.06, 0.0

LORMA_R, LORMA_ALPHA = 8, 8
ADAPTER_DIM = 32
PARTIAL_LAYERS = 4
STUDENT_KEEP = [0, 1, 3, 4, 6, 7, 9, 10, 11]   # KD student keeps 9 of 12 Phase-1 layers
KD_T, KD_ALPHA = 2.0, 0.5

HP = {   # starting points
    "full":     dict(epochs=3, lr=2e-5),
    "lorma":    dict(epochs=5, lr=3e-4),
    "lora":     dict(epochs=5, lr=3e-4),
    "adapter":  dict(epochs=5, lr=3e-4),
    "partial":  dict(epochs=4, lr=1e-4),
    "normtune": dict(epochs=8, lr=1e-3),
    "kd":       dict(epochs=3, lr=3e-5),
}
for _t in TECHNIQUES:
    assert _t in HP, _t
if "kd" in TECHNIQUES and TEACHER_TECH in TECHNIQUES:
    assert TECHNIQUES.index(TEACHER_TECH) < TECHNIQUES.index("kd"), "put the teacher technique before 'kd'"

ROOT_OUT = "/kaggle/working"
def out_dir_for(tech):
    return f"{ROOT_OUT}/cs781_phase7_arch1_{tech}_{EMB_MODE}{INFER_CHUNKS}_s{SEED}"
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)
print("Will run:", TECHNIQUES, f"| train chunks {TRAIN_CHUNKS}, infer chunks {INFER_CHUNKS} ({EMB_MODE}) | seed {SEED}")

Will run: ['lorma'] | train chunks 2, infer chunks 4 (spread) | seed 42


In [3]:


# %% [3] Tokenizer (Phase-1 BPE), cached; documents are stored as uint16 arrays to save RAM
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
VOCAB_SIZE = len(vocab)
assert VOCAB_SIZE < 65536
EOW = "\u0100"

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

@functools.lru_cache(maxsize=None)
def encode_word_cached(word):
    return tuple(token_to_id.get(sym, UNK_ID) for sym in bpe_encode_word(word))


def encode_words(words):
    ids = []
    for w in words:
        ids.extend(encode_word_cached(w))
    return ids

def encode_doc(text):
    words = str(text).split()
    if len(words) > MAX_DOC_WORDS:
        words = words[:HEAD_WORDS] + words[-(MAX_DOC_WORDS - HEAD_WORDS):]
    return np.asarray(encode_words(words), dtype=np.uint16)

In [4]:
# %% [4] Load Phase-7 data, statistics, BM25 baseline + hard negatives
def hf_local(path):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)

def parse_cases(raw):
    """-> ordered dict id -> {"text": str, "rel": list|None}. Accepts {id: {text:[sentences], relevant_candidates:[...]}} or a list of records."""
    if isinstance(raw, dict) and "data" in raw and isinstance(raw["data"], list):
        raw = raw["data"]
    if isinstance(raw, dict):
        items = list(raw.items())
    else:
        items = [(r.get("id") or r.get("case_id") or r.get("query_id") or r.get("candidate_id"), r) for r in raw]
    out = {}
    for k, v in items:
        rel = None
        if isinstance(v, dict):
            text = v.get("text") or v.get("document") or ""
            rel = v.get("relevant_candidates")
        else:
            text = v
        if isinstance(text, list):
            text = " ".join(str(s) for s in text)
        out[str(k)] = {"text": " ".join(str(text).split()), "rel": None if rel is None else [str(x) for x in rel if str(x).strip()]}
    return out

DATA = {}
for sp_name in ["train", "dev", "test"]:
    raw_q = json.load(open(hf_local(FILES[sp_name]), encoding="utf-8"))
    raw_c = json.load(open(hf_local(FILES[sp_name + "_c"]), encoding="utf-8"))
    q, c = parse_cases(raw_q), parse_cases(raw_c)
    DATA[sp_name] = {"q": q, "c": c}
    first = next(iter(raw_q.values())) if isinstance(raw_q, dict) else raw_q[0]
    print(f"{sp_name}: {len(q)} queries | {len(c)} candidates | keys of one query record: {list(first.keys()) if isinstance(first, dict) else type(first).__name__}")
    if sp_name != "test":
        rels = [len(v["rel"]) for v in q.values() if v["rel"] is not None]
        inpool = np.mean([all(r in c for r in v["rel"]) for v in q.values() if v["rel"]])
        print(f"   relevant per query: mean {np.mean(rels):.2f} median {int(np.median(rels))} max {max(rels)} | queries with 0 relevant: {sum(r == 0 for r in rels)} | all relevant ids inside the pool: {inpool:.1%}")
    else:
        print("   test queries carry labels?", any(v["rel"] for v in q.values()), "(must stay unused)")
    print(f"   query ids also present in the candidate pool: {len(set(q) & set(c))}")
    wq = np.array([len(v["text"].split()) for v in q.values()]); wc = np.array([len(v["text"].split()) for v in c.values()])
    print(f"   words per query: median {int(np.median(wq))} p90 {int(np.percentile(wq, 90))} | per candidate: median {int(np.median(wc))} p90 {int(np.percentile(wc, 90))}")

# token arrays (uint16) for every case
t0 = time.time()
TOK = {s: {"q": {k: encode_doc(v["text"]) for k, v in d["q"].items()}, "c": {k: encode_doc(v["text"]) for k, v in d["c"].items()}} for s, d in DATA.items()}
print(f"tokenised all cases in {time.time()-t0:.0f}s")
IDS = {s: {"q": list(d["q"].keys()), "c": list(d["c"].keys())} for s, d in DATA.items()}

# ---- BM25 (classical, competition data only) ----
def bm25_matrix(query_texts, doc_texts, k1=1.5, b=0.75):
    cv = CountVectorizer(lowercase=True, stop_words="english", token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b", min_df=2, max_df=0.8, dtype=np.float32)
    D = cv.fit_transform(doc_texts).tocsr(); Q = cv.transform(query_texts).tocsr()
    nd = D.shape[0]; dl = np.asarray(D.sum(1)).ravel(); avg = max(dl.mean(), 1.0)
    df = np.asarray((D > 0).sum(0)).ravel(); idf = np.log((nd - df + 0.5) / (df + 0.5) + 1.0)
    Dc = D.tocoo()
    w = Dc.data * (k1 + 1) / (Dc.data + k1 * (1 - b + b * dl[Dc.row] / avg)) * idf[Dc.col]
    W = sp.csr_matrix((w, (Dc.row, Dc.col)), shape=D.shape)
    Q.data = 1.0 + np.log(Q.data)
    return np.asarray((Q @ W.T).todense(), dtype=np.float32)

def top_k(sims, q_ids, c_ids, k=K):
    """sims (nq, nc); a candidate equal to the query id is never returned."""
    out = {}
    cidx = {c: j for j, c in enumerate(c_ids)}
    for i, q in enumerate(q_ids):
        s = sims[i].astype(np.float32).copy()
        if q in cidx:
            s[cidx[q]] = -1e9
        top = np.argsort(-s)[:k]
        out[q] = [c_ids[j] for j in top]
    return out

def micro_prf(gold, pred):
    """exactly the official Micro-F1@7 / P@7 / R@7 (tp, fp, fn summed over queries)."""
    tp = fp = fn = 0
    for q, labels in gold.items():
        g, p = set(labels), set(pred[q][:K])
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return f1, prec, rec

GOLD = {s: {q: v["rel"] for q, v in DATA[s]["q"].items()} for s in ["train", "dev"]}
t0 = time.time()
BM25 = {s: bm25_matrix([DATA[s]["q"][q]["text"] for q in IDS[s]["q"]], [DATA[s]["c"][c]["text"] for c in IDS[s]["c"]]) for s in ["train", "dev", "test"]}
print(f"BM25 matrices in {time.time()-t0:.0f}s")
for s in ["train", "dev"]:
    f1, p, r = micro_prf(GOLD[s], top_k(BM25[s], IDS[s]["q"], IDS[s]["c"]))
    print(f"BM25 baseline on {s}: Micro-F1@7 {f1:.4f} (P {p:.4f} R {r:.4f})")

# hard negatives for training queries: top BM25 candidates that are NOT relevant (and not the query itself)
HARD = {}
for i, q in enumerate(IDS["train"]["q"]):
    rel = set(GOLD["train"][q] or []); order = np.argsort(-BM25["train"][i])
    HARD[q] = [IDS["train"]["c"][j] for j in order[: HARD_POOL * 3] if IDS["train"]["c"][j] not in rel and IDS["train"]["c"][j] != q][:HARD_POOL]
TRAIN_Q = [q for q in IDS["train"]["q"] if GOLD["train"][q] and any(r in DATA["train"]["c"] for r in GOLD["train"][q])]
print(f"training queries with >=1 relevant candidate in the pool: {len(TRAIN_Q)} / {len(IDS['train']['q'])}")

# reference file (NOT one of the required experiments): BM25 only
os.makedirs(f"{ROOT_OUT}/bm25_ref", exist_ok=True)
json.dump(top_k(BM25["test"], IDS["test"]["q"], IDS["test"]["c"]), open(f"{ROOT_OUT}/bm25_ref/prediction.json", "w"))
with zipfile.ZipFile(f"{ROOT_OUT}/bm25_ref/submission_phase7_bm25_reference.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(f"{ROOT_OUT}/bm25_ref/prediction.json", arcname="prediction.json")
print("wrote the BM25 reference zip (a lexical baseline, not a model experiment)")


# %% [5] Chunking and bi-encoder embedding
def chunk_starts(n, C):
    if n <= C:
        return [0]
    st = list(range(0, n - C + 1, C))
    if st[-1] != n - C:
        st.append(n - C)
    return st

def chunks_for(toks, n, mode):
    st = chunk_starts(len(toks), CHUNK_TOK)
    if len(st) > n:
        if mode == "head":
            st = st[:n]
        else:
            st = [st[i] for i in sorted({int(round(x)) for x in np.linspace(0, len(st) - 1, n)})]
    return [np.concatenate(([BOS_ID], toks[s:s + CHUNK_TOK])).astype(np.int64) for s in st]

def pad_chunks(chunks):
    T = max(len(c) for c in chunks)
    ids = torch.full((len(chunks), T), PAD_ID, dtype=torch.long)
    lens = torch.zeros(len(chunks), dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.from_numpy(c); lens[i] = len(c)
    return ids, lens

def embed_group(model, tok_list, n, mode):
    """-> (len(tok_list), d) L2-normalised case vectors (differentiable)."""
    chunks, owner = [], []
    for i, t in enumerate(tok_list):
        cs = chunks_for(t, n, mode); chunks += cs; owner += [i] * len(cs)
    ids, lens = pad_chunks(chunks)
    ids, lens = ids.to(device), lens.to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
        h = model.forward_hidden(ids)
    T = h.size(1)
    mask = (torch.arange(T, device=device)[None, :] < lens[:, None]).unsqueeze(-1).float()
    cv = (h.float() * mask).sum(1) / lens.unsqueeze(-1).float()                 # (Nc, d) mean of token states
    own = torch.tensor(owner, device=device)
    doc = torch.zeros(len(tok_list), cv.size(1), device=device).index_add(0, own, cv)
    doc = doc / torch.bincount(own, minlength=len(tok_list)).unsqueeze(-1).float()
    return F.normalize(doc, dim=-1)

@torch.no_grad()
def embed_docs(model, tok_list, n=None, mode=None):
    """embeds many cases without grad; groups cases so that <= EVAL_CHUNK_BATCH chunks go through the model at once."""
    model.eval(); n = INFER_CHUNKS if n is None else n; mode = EMB_MODE if mode is None else mode
    outs, group, cnt = [], [], 0
    for t in tok_list:
        k = min(n, len(chunk_starts(len(t), CHUNK_TOK)))
        if group and cnt + k > EVAL_CHUNK_BATCH:
            outs.append(embed_group(model, group, n, mode).cpu()); group, cnt = [], 0
        group.append(t); cnt += k
    if group:
        outs.append(embed_group(model, group, n, mode).cpu())
    return torch.cat(outs, 0)


# %% [6] Model: Arch #1 backbone + PEFT modules (no extra head: the case vector is the pooled backbone state)
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def build_rope_cache(seq_len, head_dim, device, base=10000.0, scale=1.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float() * scale      # scale < 1 = position interpolation
    freqs = torch.einsum("i,j->ij", t, inv_freq)
    emb = torch.cat((freqs, freqs), dim=-1)
    return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class LoRMAPlus(nn.Module):
    """h' = h + (alpha/r) B(A h) = ((alpha/r) B A + I) W0 x with h = W0 x (multiplicative; B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, h):
        return h + self.scale * F.linear(F.linear(h, self.A), self.B)

class LoRAAdapter(nn.Module):
    """Additive update (alpha/r) B(A x), added to W0 x (LoRA; B = 0 at init)."""
    def __init__(self, d, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.scale * F.linear(F.linear(x, self.A), self.B)

class BottleneckAdapter(nn.Module):
    """Houlsby-style adapter: h + up(gelu(down(h))); `up` starts at zero (identity at init)."""
    def __init__(self, d, m):
        super().__init__()
        self.down = nn.Linear(d, m)
        self.up = nn.Linear(m, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head, self.head_dim, self.n_embd = n_head, n_embd // n_head, n_embd
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
        self.adapt_mode = None
        self.q_ad = None
        self.v_ad = None
    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)
        if self.adapt_mode == "lorma":
            q, v = self.q_ad(q), self.v_ad(v)
        elif self.adapt_mode == "lora":
            q, v = q + self.q_ad(x), v + self.v_ad(x)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos[:, :, :T, :], sin[:, :, :T, :])
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.proj(out.transpose(1, 2).contiguous().view(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, mult, dropout):
        super().__init__()
        hidden = ((int(mult * n_embd) + 63) // 64) * 64
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(hidden, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.w3(F.silu(self.w1(x)) * self.w2(x)))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, ffn_mult, dropout):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = RMSNorm(n_embd)
        self.ffn = SwiGLU(n_embd, ffn_mult, dropout)
        self.ad1 = None
        self.ad2 = None
    def forward(self, x, cos, sin):
        a = self.attn(self.ln1(x), cos, sin)
        if self.ad1 is not None:
            a = self.ad1(a)
        x = x + a
        f = self.ffn(self.ln2(x))
        if self.ad2 is not None:
            f = self.ad2(f)
        return x + f

class LegalLLaMA(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, block_size, ffn_mult, dropout, rope_scale=1.0):
        super().__init__()
        self.block_size, self.head_dim = block_size, n_embd // n_head
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, ffn_mult, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        cos, sin = build_rope_cache(block_size, self.head_dim, device="cpu", scale=rope_scale)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward_hidden(self, idx):
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope_cos.to(idx.device), self.rope_sin.to(idx.device)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.ln_f(x)

ckpt = torch.load(MODEL_CKPT_PATH, map_location="cpu")
cfg = ckpt["config"]
assert cfg["vocab_size"] == VOCAB_SIZE, (cfg["vocab_size"], VOCAB_SIZE)
assert CHUNK_TOK + 1 <= cfg["block_size"]
print(f"Phase-1 model: {cfg['n_layer']} layers, d={cfg['n_embd']}, context {cfg['block_size']}")

def make_student_state(sd, keep):
    new = {}
    for k, v in sd.items():
        if k.startswith("blocks."):
            parts = k.split(".")
            old = int(parts[1])
            if old in keep:
                parts[1] = str(keep.index(old))
                new[".".join(parts)] = v
        else:
            new[k] = v
    return new

def build_model(technique, keep_layers=None):
    sd, n_layer = ckpt["model_state_dict"], cfg["n_layer"]
    if keep_layers is not None:
        sd, n_layer = make_student_state(sd, keep_layers), len(keep_layers)
    bb = LegalLLaMA(cfg["vocab_size"], n_layer, cfg["n_head"], cfg["n_embd"], cfg["block_size"], cfg["ffn_mult"], DROPOUT_FT)
    bb.load_state_dict(sd)
    d = cfg["n_embd"]
    if technique in ("lorma", "lora", "adapter", "partial", "normtune"):
        for p in bb.parameters():
            p.requires_grad = False
        if technique in ("lorma", "lora"):
            for blk in bb.blocks:
                blk.attn.adapt_mode = technique
                make = LoRMAPlus if technique == "lorma" else LoRAAdapter
                blk.attn.q_ad, blk.attn.v_ad = make(d, LORMA_R, LORMA_ALPHA), make(d, LORMA_R, LORMA_ALPHA)
        elif technique == "adapter":
            for blk in bb.blocks:
                blk.ad1, blk.ad2 = BottleneckAdapter(d, ADAPTER_DIM), BottleneckAdapter(d, ADAPTER_DIM)
        elif technique == "partial":
            for blk in bb.blocks[-PARTIAL_LAYERS:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in bb.ln_f.parameters():
                p.requires_grad = True
        elif technique == "normtune":
            for m in bb.modules():
                if isinstance(m, RMSNorm):
                    m.weight.requires_grad = True
    bb = bb.to(device)
    n_tot = sum(p.numel() for p in bb.parameters()); n_trn = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    print(f"[{technique}{' student' if keep_layers else ''}] trainable {n_trn/1e6:.3f}M of {n_tot/1e6:.1f}M params ({100*n_trn/n_tot:.2f}%)")
    return bb


# %% [7] Retrieval evaluation helpers
def sims_for(model, split):
    """cosine similarity matrix (n_queries, n_candidates) of a split; also returns the embedding time."""
    t0 = time.time()
    qe = embed_docs(model, [TOK[split]["q"][k] for k in IDS[split]["q"]])
    ce = embed_docs(model, [TOK[split]["c"][k] for k in IDS[split]["c"]])
    return (qe @ ce.T).numpy(), time.time() - t0

def eval_dev(model):
    S, sec = sims_for(model, "dev")
    f1, p, r = micro_prf(GOLD["dev"], top_k(S, IDS["dev"]["q"], IDS["dev"]["c"]))
    return f1, p, r, S, sec

def zscore_rows(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)


# %% [8] Training: multi-positive InfoNCE with BM25 hard negatives (+ optional similarity distillation)
def fit(model, epochs, lr, best_path, teacher=None):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    steps_per_epoch = math.ceil(len(TRAIN_Q) / ACCUM); total = epochs * steps_per_epoch
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_factor(step):
        if step < warm:
            return (step + 1) / warm
        prog = (step - warm) / max(1, total - warm)
        return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))
    rng = random.Random(SEED)
    pool_ids = IDS["train"]["c"]
    f1_0, p0, r0, _, sec0 = eval_dev(model)
    print(f"[epoch 0] (no training) dev Micro-F1@7 {f1_0:.4f} (P {p0:.4f} R {r0:.4f}) | eval {sec0:.0f}s   <- Phase-1 representation transfer")
    best, step, t0, history, eta = -1.0, 0, time.time(), [{"epoch": 0, "dev_f1": f1_0}], False
    torch.save({"model": model.state_dict(), "epoch": 0, "dev_f1": f1_0}, best_path); best = f1_0
    if teacher is not None:
        teacher.eval()
    for epoch in range(1, epochs + 1):
        model.train()
        order = TRAIN_Q[:]; rng.shuffle(order)
        run, n_b = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for bi, q in enumerate(order):
            rel = [r for r in GOLD["train"][q] if r in DATA["train"]["c"]]
            pos = rng.sample(rel, min(N_POS, len(rel)))
            hard = rng.sample(HARD[q], min(N_HARD, len(HARD[q])))
            relset = set(rel)
            rnd = []
            while len(rnd) < N_RAND:
                c = rng.choice(pool_ids)
                if c not in relset and c != q and c not in rnd and c not in hard:
                    rnd.append(c)
            toks = [TOK["train"]["q"][q]] + [TOK["train"]["c"][c] for c in pos + hard + rnd]
            D = embed_group(model, toks, TRAIN_CHUNKS, EMB_MODE)
            s = (D[1:] @ D[0]) / TAU
            sp_, sn_ = s[:len(pos)], s[len(pos):]
            loss = (torch.logaddexp(sp_, torch.logsumexp(sn_, 0)) - sp_).mean()
            if teacher is not None:
                with torch.no_grad():
                    Dt = embed_group(teacher, toks, TRAIN_CHUNKS, EMB_MODE)
                    st_ = (Dt[1:] @ Dt[0]) / TAU
                kd = F.kl_div(F.log_softmax(s / KD_T, dim=0), F.softmax(st_ / KD_T, dim=0), reduction="sum") * (KD_T ** 2)
                loss = (1 - KD_ALPHA) * loss + KD_ALPHA * kd
            scaler.scale(loss / ACCUM).backward()
            run += loss.item(); n_b += 1
            if (bi + 1) % ACCUM == 0 or bi + 1 == len(order):
                for g in optimizer.param_groups:
                    g["lr"] = lr * lr_factor(step)
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); step += 1
            if not eta and bi == 19:
                per = (time.time() - t0) / 20
                print(f"  est. epoch time {per*len(order)/60:.1f} min ({len(order)} queries, {per:.2f}s/query) -> {epochs} epochs ~ {epochs*per*len(order)/60:.0f} min (+ dev evals)")
                eta = True
        f1, p, r, _, sec = eval_dev(model)
        history.append({"epoch": epoch, "train_loss": run / max(1, n_b), "dev_f1": f1})
        print(f"[epoch {epoch}] loss {run/max(1,n_b):.4f} | dev Micro-F1@7 {f1:.4f} (P {p:.4f} R {r:.4f}) | {(time.time()-t0)/60:.1f}m")
        if f1 > best:
            best = f1
            torch.save({"model": model.state_dict(), "epoch": epoch, "dev_f1": f1}, best_path)
            print("  -> new best checkpoint saved")
    model.load_state_dict(torch.load(best_path, map_location="cpu")["model"])
    return history


# %% [9] One technique end-to-end (train -> dev report -> test ranking -> save similarities)
def set_seed(s):
    random.seed(s); torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def run_technique(tech):
    out_dir = out_dir_for(tech); os.makedirs(out_dir, exist_ok=True)
    best_path, pred_path = os.path.join(out_dir, "pcr_best.pt"), os.path.join(out_dir, "prediction.json")
    summ_path, sims_path = os.path.join(out_dir, "summary.json"), os.path.join(out_dir, "sims.pkl")
    if SKIP_DONE and all(os.path.exists(p) for p in [pred_path, summ_path, sims_path]):
        print(f"\n##### [{tech}] already finished -> skipping"); return json.load(open(summ_path))
    print(f"\n{'='*78}\n##### Phase 7 | Arch1 | {tech} | {EMB_MODE} {TRAIN_CHUNKS}->{INFER_CHUNKS} chunks | seed {SEED} | {HP[tech]}\n{'='*78}")
    set_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_run = time.time(); teacher = None
    if tech == "kd":
        teacher = build_model(TEACHER_TECH)
        t_ckpt = TEACHER_CKPT or os.path.join(out_dir_for(TEACHER_TECH), "pcr_best.pt")
        if os.path.exists(t_ckpt):
            teacher.load_state_dict(torch.load(t_ckpt, map_location="cpu")["model"]); print("Loaded teacher from", t_ckpt)
        else:
            print(f"No saved teacher found -> training the teacher ({TEACHER_TECH}) first")
            fit(teacher, best_path=os.path.join(out_dir, "teacher_best.pt"), **HP[TEACHER_TECH])
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
        print(f"TEACHER dev Micro-F1@7: {eval_dev(teacher)[0]:.4f}")
        model = build_model("full", keep_layers=STUDENT_KEEP)
        t_p, s_p = sum(p.numel() for p in teacher.parameters()), sum(p.numel() for p in model.parameters())
        print(f"Teacher {t_p/1e6:.1f}M -> student {s_p/1e6:.1f}M params ({100*(1-s_p/t_p):.0f}% smaller)")
        history = fit(model, best_path=best_path, teacher=teacher, **HP["kd"])
    else:
        model = build_model(tech)
        history = fit(model, best_path=best_path, **HP[tech])
    json.dump(history, open(os.path.join(out_dir, "history.json"), "w"), indent=2)
    train_min = (time.time() - t_run) / 60
    print(f"[{tech}] training finished in {train_min:.1f} min")

    f1, p, r, S_dev, sec_dev = eval_dev(model)
    print(f"[{tech}] DEV (best epoch): Micro-F1@7 {f1:.4f} | P@7 {p:.4f} | R@7 {r:.4f}")
    S_test, sec_test = sims_for(model, "test")
    pred = top_k(S_test, IDS["test"]["q"], IDS["test"]["c"])
    assert set(pred) == set(IDS["test"]["q"]) and all(len(v) == K == len(set(v)) and all(c in DATA["test"]["c"] for c in v) for v in pred.values())
    json.dump(pred, open(pred_path, "w"))
    pickle.dump({"dev_q": IDS["dev"]["q"], "dev_c": IDS["dev"]["c"], "dev": S_dev.astype(np.float16),
                 "test_q": IDS["test"]["q"], "test_c": IDS["test"]["c"], "test": S_test.astype(np.float16)}, open(sims_path, "wb"))
    n_tot = sum(p.numel() for p in model.parameters()); n_trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    summary = {"technique": tech, "dev_f1": f1, "dev_p": p, "dev_r": r, "train_minutes": round(train_min, 1),
               "embed_sec_dev": round(sec_dev, 1), "embed_sec_test": round(sec_test, 1),
               "params_total_M": round(n_tot / 1e6, 2), "params_trainable_M": round(n_trn / 1e6, 3),
               "peak_gpu_mem_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else None}
    json.dump(summary, open(summ_path, "w"), indent=2)
    del model, teacher
    return summary


# %% [10] Run everything chosen in cell [2]
all_results = []
for tech in TECHNIQUES:
    try:
        res = run_technique(tech)
    except Exception:
        print(f"\n!!!!! [{tech}] FAILED -- continuing")
        traceback.print_exc()
        res = {"technique": tech, "error": True}
    all_results.append(res)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print("\n" + "=" * 78 + "\nSUMMARY (dev Micro-F1@7)\n" + "=" * 78)
for r in all_results:
    if r.get("error"):
        print(f"{r['technique']:9s} FAILED")
    else:
        print(f"{r['technique']:9s} dev F1 {r['dev_f1']:.4f} (P {r['dev_p']:.4f} R {r['dev_r']:.4f}) | train {r['train_minutes']} min | embed dev {r['embed_sec_dev']}s test {r['embed_sec_test']}s | "
              f"params {r['params_total_M']}M (trainable {r['params_trainable_M']}M) | mem {r['peak_gpu_mem_GB']} GB")


# %% [11] Zips: one Codabench submission zip per run (prediction.json at the zip root) + a results zip (similarities for fusion)
SUB_DIR, RES_DIR = f"{ROOT_OUT}/submissions", f"{ROOT_OUT}/results_zips"
os.makedirs(SUB_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)
for t in TECHNIQUES:
    d = out_dir_for(t)
    if not os.path.exists(f"{d}/prediction.json"):
        print(f"[{t}] prediction.json not found -> skipped"); continue
    tag = f"phase7_arch1_{t}_{EMB_MODE}{INFER_CHUNKS}_s{SEED}"
    with zipfile.ZipFile(f"{SUB_DIR}/submission_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(f"{d}/prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(f"{RES_DIR}/results_{tag}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for fn in ["prediction.json", "sims.pkl", "history.json", "summary.json"]:
            if os.path.exists(f"{d}/{fn}"):
                zf.write(f"{d}/{fn}", arcname=fn)
    print(f"[{t}] submission_{tag}.zip")
print("Upload the zips in", SUB_DIR, "to Codabench.")


# %% [12] OPTIONAL fusion on CPU: average the saved similarities of several runs and/or mix in BM25. Needs cells [1]-[4] and [7].
# Upload each run's sims.pkl (renamed, e.g. sims_lorma.pkl) as a dataset and list them here. {} = skip.
SIM_MEMBERS = {   # name: path to sims.pkl
    # "lorma": "/kaggle/input/YOUR_DS/sims_lorma.pkl",
    # "full":  "/kaggle/input/YOUR_DS/sims_full.pkl",
    # "kd":    "/kaggle/input/YOUR_DS/sims_kd.pkl",
}
ALPHAS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0]     # weight of the model similarity (rest = BM25)
if SIM_MEMBERS:
    sims = {n: pickle.load(open(p, "rb")) for n, p in SIM_MEMBERS.items()}
    for n, s in sims.items():
        assert s["dev_q"] == IDS["dev"]["q"] and s["dev_c"] == IDS["dev"]["c"] and s["test_q"] == IDS["test"]["q"] and s["test_c"] == IDS["test"]["c"], f"{n}: id order differs"
    def model_mix(split, use):
        return np.mean([zscore_rows(sims[n][split].astype(np.float32)) for n in use], axis=0)
    def f1_of(split_S, use_names=None):
        return micro_prf(GOLD["dev"], top_k(split_S, IDS["dev"]["q"], IDS["dev"]["c"]))[0]
    names = list(sims)
    for n in names:
        print(f"member {n:10s} dev Micro-F1@7 {f1_of(zscore_rows(sims[n]['dev'].astype(np.float32))):.4f}")
    bm_dev, bm_test = zscore_rows(BM25["dev"]), zscore_rows(BM25["test"])
    print(f"BM25 alone       dev Micro-F1@7 {f1_of(bm_dev):.4f}")
    best = (-1, None, None)
    for use in [list(c) for r in range(1, len(names) + 1) for c in itertools.combinations(names, r)]:
        Md = model_mix("dev", use)
        row = {a: f1_of(a * Md + (1 - a) * bm_dev) for a in ALPHAS}
        a_best = max(row, key=row.get)
        print(f"{'+'.join(use):28s} model-only {row[1.0]:.4f} | best alpha {a_best} -> {row[a_best]:.4f}")
        if row[a_best] > best[0]:
            best = (row[a_best], use, a_best)
    print("BEST on dev (118 queries -> noisy, differences < ~0.01 mean little):", best)
    _, use, alpha = best
    S_test = alpha * model_mix("test", use) + (1 - alpha) * bm_test
    pred = top_k(S_test, IDS["test"]["q"], IDS["test"]["c"])
    assert all(len(v) == K == len(set(v)) for v in pred.values())
    ED = f"{ROOT_OUT}/phase7_fusion"; os.makedirs(ED, exist_ok=True)
    json.dump(pred, open(f"{ED}/prediction.json", "w"))
    with zipfile.ZipFile(f"{ED}/submission_phase7_fusion.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(f"{ED}/prediction.json", arcname="prediction.json")
    print("wrote", f"{ED}/submission_phase7_fusion.zip", "| members", use, "| alpha", alpha)

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train: 827 queries | 4320 candidates | keys of one query record: ['text', 'relevant_candidates']
   relevant per query: mean 6.42 median 4 max 52 | queries with 0 relevant: 10 | all relevant ids inside the pool: 100.0%
   query ids also present in the candidate pool: 827
   words per query: median 4508 p90 12602 | per candidate: median 3632 p90 12422


dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (3 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test: 237 queries | 1727 candidates | keys of one query record: ['text']
   test queries carry labels? False (must stay unused)
   query ids also present in the candidate pool: 237
   words per query: median 4653 p90 14064 | per candidate: median 3947 p90 13490
tokenised all cases in 24s
BM25 matrices in 27s
BM25 baseline on train: Micro-F1@7 0.1566 (P 0.1501 R 0.1637)
BM25 baseline on dev: Micro-F1@7 0.1845 (P 0.2010 R 0.1706)
training queries with >=1 relevant candidate in the pool: 817 / 827
wrote the BM25 reference zip (a lexical baseline, not a model experiment)
Phase-1 model: 12 layers, d=576, context 1024

##### Phase 7 | Arch1 | lorma | spread 2->4 chunks | seed 42 | {'epochs': 5, 'lr': 0.0003}
[lorma] trainable 0.221M of 66.4M params (0.33%)
[epoch 0] (no training) dev Micro-F1@7 0.2802 (P 0.3051 R 0.2590) | eval 55s   <- Phase-1 representation transfer
  est. epoch time 9.4 min (817 queries, 0.69s/query) -> 5 epochs ~ 47 min (+ dev evals)
[epoch 1] loss 1.7410 | dev Micro-F1@